# FT-Transformer

In [1]:
!python -m pip install --upgrade pip setuptools wheel

Defaulting to user installation because normal site-packages is not writeable
Could not fetch URL https://pypi.org/simple/pip/: There was a problem confirming the ssl certificate: HTTPSConnectionPool(host='pypi.org', port=443): Max retries exceeded with url: /simple/pip/ (Caused by SSLError(SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:1129)'))) - skipping


In [2]:
!pip install ipywidgets

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------- ----- 786.4/914.9 kB 6.7 MB/s eta 0:00:01
   ---------------------------------------- 914.9/914.9 kB 5.9 MB/s  0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 2.2/2.2 MB 17.7 MB/s  0:00:00

   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   ---------------------------------------- 3/3 [ipywidgets]



In [3]:
!pip install rtdl_revisiting_models -q

In [4]:
import pandas as pd
import numpy as np
import os

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from rtdl_revisiting_models import FTTransformer

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

DATA_PATH = r"D:\墨大sml作业\FeatureA_Repeated"
OUTPUT_PATH = r"D:\墨大sml作业\Official_FTTransformer_FeatureA_Results"

os.makedirs(OUTPUT_PATH, exist_ok=True)

N_REPEATS = 10

cpu



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "d:\Program Files\Python39\lib\runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "d:\Program Files\Python39\lib\runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "C:\Users\MJ\AppData\Roaming\Python\Python39\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\MJ\AppData\Roaming\Python\Python39\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Us

In [6]:
feature_cols = [
    "user_avg_rating",
    "user_rating_count",
    "user_rating_std",
    "user_like_count",
    "user_like_ratio",
    "user_rating_timespan",
    "user_avg_gap_days",
    "item_avg_rating",
    "item_rating_count",
    "item_rating_std",
    "item_like_count",
    "item_like_ratio",
    "global_mean",
    "movie_age_at_rating"
]

In [7]:
class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [8]:
def compute_auc_from_scratch(y_true, y_score):
    y_true = np.array(y_true)
    y_score = np.array(y_score)

    sorted_indices = np.argsort(-y_score)
    y_true_sorted = y_true[sorted_indices]

    pos_count = np.sum(y_true == 1)
    neg_count = np.sum(y_true == 0)

    if pos_count == 0 or neg_count == 0:
        return 0

    tp = 0
    fp = 0

    tpr_list = [0]
    fpr_list = [0]

    for label in y_true_sorted:
        if label == 1:
            tp += 1
        else:
            fp += 1

        tpr_list.append(tp / pos_count)
        fpr_list.append(fp / neg_count)

    auc = 0

    for i in range(1, len(tpr_list)):
        auc += (
            (fpr_list[i] - fpr_list[i - 1])
            * (tpr_list[i] + tpr_list[i - 1])
            / 2
        )

    return auc


def compute_metrics_from_scratch(y_true, y_pred, y_score):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    auc = compute_auc_from_scratch(y_true, y_score)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn
    }

In [9]:
def stratified_sample_binary(df, sample_size, random_seed):
    pos_df = df[df["label"] == 1]
    neg_df = df[df["label"] == 0]

    pos_n = sample_size // 2
    neg_n = sample_size - pos_n

    pos_sample = pos_df.sample(
        n=pos_n,
        random_state=random_seed
    )

    neg_sample = neg_df.sample(
        n=neg_n,
        random_state=random_seed
    )

    sampled_df = pd.concat(
        [pos_sample, neg_sample],
        axis=0
    ).sample(
        frac=1,
        random_state=random_seed
    ).reset_index(drop=True)

    return sampled_df

In [10]:
def train_official_ft_transformer(
    train_df,
    test_df,
    lr=1e-4,
    weight_decay=1e-5,
    batch_size=4096,
    epochs=3
):
    X_train = train_df[feature_cols].values
    y_train = train_df["label"].values

    X_test = test_df[feature_cols].values
    y_test = test_df["label"].values

    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    train_dataset = TabularDataset(X_train, y_train)
    test_dataset = TabularDataset(X_test, y_test)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    model = FTTransformer(
        n_cont_features=len(feature_cols),
        cat_cardinalities=[],
        d_out=1,
        **FTTransformer.get_default_kwargs()
    ).to(device)

    optimizer = model.make_default_optimizer()
    
    # Override default optimizer lr / weight_decay if needed
    for group in optimizer.param_groups:
        group["lr"] = lr
        group["weight_decay"] = weight_decay

    criterion = nn.BCEWithLogitsLoss()

    model.train()

    for epoch in range(epochs):
        total_loss = 0

        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()

            logits = model(batch_X, None).squeeze(1)

            loss = criterion(logits, batch_y)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch + 1} loss:", round(total_loss, 4))

    model.eval()

    all_preds = []
    all_scores = []
    all_labels = []

    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X = batch_X.to(device)

            logits = model(batch_X, None).squeeze(1)
            probs = torch.sigmoid(logits)

            preds = (probs >= 0.5).int()

            all_preds.extend(preds.detach().cpu().tolist())
            all_scores.extend(probs.detach().cpu().tolist())
            all_labels.extend(batch_y.detach().cpu().tolist())

    metrics = compute_metrics_from_scratch(
        all_labels,
        all_preds,
        all_scores
    )

    return metrics

In [11]:
TRAIN_SAMPLE_SIZE = 200000
TEST_SAMPLE_SIZE = 100000

LR_VALUES = [5e-5, 1e-4, 5e-4]
WEIGHT_DECAY_VALUES = [1e-5]

BATCH_SIZE = 4096
EPOCHS = 3

all_results = []

for repeat_id in range(1, N_REPEATS + 1):
    print("=" * 60)
    print(f"Repeat {repeat_id}")
    print("=" * 60)

    repeat_folder = os.path.join(
        DATA_PATH,
        f"repeat_{repeat_id:02d}"
    )

    train_path = os.path.join(repeat_folder, "feature_A_train.csv")
    test_path = os.path.join(repeat_folder, "feature_A_test.csv")

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    train_df = stratified_sample_binary(
        train_df,
        sample_size=TRAIN_SAMPLE_SIZE,
        random_seed=42 + repeat_id
    )

    test_df = stratified_sample_binary(
        test_df,
        sample_size=TEST_SAMPLE_SIZE,
        random_seed=100 + repeat_id
    )

    print("Sampled train shape:", train_df.shape)
    print("Sampled test shape:", test_df.shape)

    best_result = None
    best_f1 = -1

    for lr in LR_VALUES:
        for weight_decay in WEIGHT_DECAY_VALUES:
            print(f"Trying lr={lr}, weight_decay={weight_decay}")

            metrics = train_official_ft_transformer(
                train_df=train_df,
                test_df=test_df,
                lr=lr,
                weight_decay=weight_decay,
                batch_size=BATCH_SIZE,
                epochs=EPOCHS
            )

            if metrics["f1"] > best_f1:
                best_f1 = metrics["f1"]

                best_result = {
                    "repeat": repeat_id,
                    "best_lr": lr,
                    "best_weight_decay": weight_decay,
                    "batch_size": BATCH_SIZE,
                    **metrics
                }

    all_results.append(best_result)
    print(best_result)

Repeat 1
Sampled train shape: (200000, 17)
Sampled test shape: (100000, 17)
Trying lr=5e-05, weight_decay=1e-05
Epoch 1 loss: 28.0861
Epoch 2 loss: 26.9791
Epoch 3 loss: 26.8692
Trying lr=0.0001, weight_decay=1e-05
Epoch 1 loss: 27.6436
Epoch 2 loss: 26.8325
Epoch 3 loss: 26.7635
Trying lr=0.0005, weight_decay=1e-05
Epoch 1 loss: 28.4805
Epoch 2 loss: 26.8204
Epoch 3 loss: 26.7221
{'repeat': 1, 'best_lr': 0.0001, 'best_weight_decay': 1e-05, 'batch_size': 4096, 'accuracy': np.float64(0.71466), 'precision': np.float64(0.7043835929466428), 'recall': np.float64(0.7398), 'f1': np.float64(0.72165752970326), 'auc': np.float64(0.7897571300000129), 'tp': np.int64(36990), 'tn': np.int64(34476), 'fp': np.int64(15524), 'fn': np.int64(13010)}
Repeat 2
Sampled train shape: (200000, 17)
Sampled test shape: (100000, 17)
Trying lr=5e-05, weight_decay=1e-05
Epoch 1 loss: 28.0778
Epoch 2 loss: 26.9909
Epoch 3 loss: 26.8667
Trying lr=0.0001, weight_decay=1e-05
Epoch 1 loss: 27.8259
Epoch 2 loss: 26.9017
E

In [12]:
results_df = pd.DataFrame(all_results)

results_path = os.path.join(
    OUTPUT_PATH,
    "Official_FTTransformer_results.csv"
)

results_df.to_csv(
    results_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved to:")
print(results_path)

results_df

Saved to:
D:\墨大sml作业\Official_FTTransformer_FeatureA_Results\Official_FTTransformer_results.csv


,repeat,best_lr,best_weight_decay,batch_size,accuracy,precision,recall,f1,auc,tp,tn,fp,fn
0,1,0.00010,0.00001,4096,0.71466,0.704384,0.73980,0.721658,0.789757,36990,34476,15524,13010
1,2,0.00050,0.00001,4096,0.71327,0.693688,0.76382,0.727067,0.787194,38191,33136,16864,11809
2,3,0.00005,0.00001,4096,0.71577,0.697820,0.76114,0.728106,0.789853,38057,33520,16480,11943
3,4,0.00005,0.00001,4096,0.71549,0.690265,0.78178,0.733178,0.791253,39089,32460,17540,10911
4,5,0.00005,0.00001,4096,0.71442,0.692741,0.77066,0.729626,0.790789,38533,32909,17091,11467
5,6,0.00010,0.00001,4096,0.71495,0.696650,0.76148,0.727624,0.789124,38074,33421,16579,11926
6,7,0.00005,0.00001,4096,0.71209,0.701833,0.73750,0.719225,0.788511,36875,34334,15666,13125
7,8,0.00050,0.00001,4096,0.71722,0.705335,0.74616,0.725173,0.791100,37308,34414,15586,12692
8,9,0.00005,0.00001,4096,0.71610,0.695001,0.77020,0.730671,0.791616,38510,33100,16900,11490
9,10,0.00010,0.00001,4096,0.71483,0.707818,0.73170,0.719561,0.789771,36585,34898,15102,13415


In [13]:
summary_records = []

for metric in ["accuracy", "precision", "recall", "f1", "auc"]:
    values = results_df[metric].values

    summary_records.append({
        "metric": metric,
        "mean": np.mean(values),
        "std": np.std(values, ddof=1),
        "standard_error": np.std(values, ddof=1) / np.sqrt(len(values))
    })

summary_df = pd.DataFrame(summary_records)

summary_path = os.path.join(
    OUTPUT_PATH,
    "Official_FTTransformer_summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig"
)

summary_df

,metric,mean,std,standard_error
0,accuracy,0.714880,0.001444,0.000457
1,precision,0.698553,0.005958,0.001884
2,recall,0.756424,0.016630,0.005259
3,f1,0.726189,0.004729,0.001495
4,auc,0.789897,0.001370,0.000433


In [16]:
# Best learning rate frequency for FT-Transformer

best_lr_frequency = (
    results_df["best_lr"]
    .value_counts()
    .reset_index()
)

best_lr_frequency.columns = ["learning_rate", "frequency"]

best_lr_frequency_path = os.path.join(
    OUTPUT_PATH,
    "FTTransformer_best_lr_frequency.csv"
)

best_lr_frequency.to_csv(
    best_lr_frequency_path,
    index=False,
    encoding="utf-8-sig"
)


best_lr_frequency

,learning_rate,frequency
0,0.00005,5
1,0.00010,3
2,0.00050,2
